<a target="_blank" href="https://colab.research.google.com/github/Tensor-Reloaded/Neural-Networks-Template-2025/blob/main/Lab02/Assignment1.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# **Assignment 1 (10 points)**

## **Solving a linear system in python**

In this homework, you will familiarize yourself with key linear algebra con-
cepts and Python programming by solving a system of linear equations. You
will explore multiple methods for solving such systems, including Cramer’s rule
and matrix inversion. By the end of this assignment, you will have a good un-
derstanding of how to represent and manipulate matrices and vectors in Python.

We begin with the following system of 3 linear equations with 3 unknowns:
$$ 2x + 3y - z = 5 $$
$$ x - y + 4z = 6 $$
$$ 3x + y + 2z = 7 $$

This system can be vectorized in the following form:
$$ A \cdot X = B $$
where:
$$
A = \begin{bmatrix}
2 & 3 & -1 \\
1 & -1 & 4 \\
3 & 1 & 2
\end{bmatrix}, \quad 
X = \begin{bmatrix}
x \\
y \\
z
\end{bmatrix}, \quad 
B = \begin{bmatrix}
5 \\
6 \\
7
\end{bmatrix}
$$

**Considerations**
- do not use any linear algebra framework such as $numpy$
- use python lists as data structures for matrices and vectors
- experiment with other values for the coefficients and free terms

### **1. Parsing the System of Equations (1 point)**

The first task is to implement a Python script that reads a system of linear equations from a text file and parses it into a matrix $A$ and a vector $B$. You will use the input format described below to extract the coefficients for $A$ and $B$.

**Input File Format**
```text
2x + 3y - z = 5
x - y + 4z = 6
3x + y + 2z = 7
```

Note that the coefficients are always in the order x, y and z and the terms are always space separated

In [19]:
import re
import pathlib

def load_system(path: pathlib.Path) -> tuple[list[list[float]], list[float]]:
    A, B = [], []
    with path.open() as f:
        for line in f:
            # print(line)
            if not line.split():
                continue
            line_content = line.split()
            coeffs = []
            isB = False
            sign = 1
            for cont in line.split():
                # print (cont)
                match cont:
                    case '-':
                        sign = -1
                    case '+':
                        sign = 1
                    case '=':
                        isB = True
                    case _:
                        if cont.find(r'[a-z]'):
                            match = re.findall(r'\d+',cont)
                            if match:
                                number = float(match[0]) * sign
                            else:
                                number = 1 * sign
                            if isB:
                                B.append(number)
                            else:
                                coeffs.append(number)
                            sign = 1
                            continue
                        coeffs.append(0.0)
            A.append(coeffs)

    return A, B

A, B = load_system(pathlib.Path("system.txt"))
print(f"{A=} {B=}")

A=[[2.0, 3.0, -1], [1, -1, 4.0], [3.0, 1, 2.0]] B=[5.0, 6.0, 7.0]


### **2. Matrix and Vector Operations (5 points)**

Once you have successfully parsed the matrix and vector, complete the following exercises to manipulate and understand basic matrix and vector operations. Write Python functions for each of these tasks:

#### 2.1. Determinant

Write a function to compute the determinant of matrix $A$. Recall one of the formulae for the determinant of a $3x3$ matrix:
$$ \text{det}(A) = a_{11}(a_{22}a_{33} - a_{23}a_{32}) - a_{12}(a_{21}a_{33} - a_{23}a_{31}) + a_{13}(a_{21}a_{32} - a_{22}a_{31}) $$

In [40]:
def determinant(matrix: list[list[float]]) -> float:
    det = 0.0
    n = len(matrix)
    if n == 3:
        for i in range(0,3):
            i1 = (i+1) % 3
            i2 = (i+2) % 3
            if i1 > i2:
                i1, i2 = i2, i1
            if i % 2 != 0:
                det = det + matrix[i][i]*(matrix[1][i1]*matrix[2][i2] - matrix[1][i2]*matrix[2][i1])
                continue
            det = det - matrix[i][i]*(matrix[1][i1]*matrix[2][i2] - matrix[1][i2]*matrix[2][i1])
    elif n == 2:
        det = matrix[0][0]*matrix[1][1] - matrix[0][1]*matrix[1][0]
    else:
        raise Exception("Matrix needs to be of size 2 or 3")
    return det

print(f"{determinant(A)=}")

determinant(A)=14.0


#### 2.2. Trace

Compute the sum of the elements along the main diagonal of matrix $A$. For a matrix $A$, this is:
$$ \text{Trace}(A) = a_{11} + a_{22} + a_{33} $$

In [24]:
def trace(matrix: list[list[float]]) -> float:
    trace = 0
    for i in range (0,3):
        trace += matrix[i][i]
    return trace

print(f"{trace(A)=}")

trace(A)=3.0


#### 2.3. Vector norm

Compute the Euclidean norm of vector $B$, which is:
$$ ||B|| = \sqrt{b_1^2 + b_2^2 + b_3^2} $$

In [25]:
import math
def norm(vector: list[float]) -> float:
    norm = 0
    for i in range(0,3):
        norm += vector[i]**2
    return norm ** (1/2)

print(f"{norm(B)=}")

norm(B)=10.488088481701515


#### 2.4. Transpose of matrix

Write a function to compute the transpose of matrix $A$. The transpose of a matrix $A$ is obtained by swapping its rows and columns.
    

In [27]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    rows, cols = len(matrix), len(matrix[0])
    AT = [[0.0 for _ in range(rows)] for _ in range(cols)]
    for i in range(rows):
        for j in range(cols):
            AT[j][i] = matrix[i][j]
    return AT

print(f"{transpose(A)=}")

transpose(A)=[[2.0, 1, 3.0], [3.0, -1, 1], [-1, 4.0, 2.0]]


#### 2.5. Matrix-vector multiplication

Write a function that multiplies matrix $A$ with vector $B$.

In [29]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float]:
    rowsA, colsA = len(matrix), len(matrix[0])
    rowsB = len(vector)
    assert rowsB == colsA, "Vector length must match matrix columns"
    y = [0.0 for _ in range(rowsA)]
    for i in range(rowsA):
        for j in range(colsA):
            y[i] += matrix[i][j] * vector[j]
    return y 

print(f"{multiply(A, B)=}")

multiply(A, B)=[21.0, 27.0, 35.0]


### **3. Solving using Cramer's Rule (1 point)**

Now that you have explored basic matrix operations, solve the system of linear equations using Cramer's rule.

**Cramer's Rule:**

Cramer's rule allows you to solve for each unknown $x$, $y$, and $z$ using determinants. For example:
$$ x = \frac{\text{det}(A_x)}{\text{det}(A)}, \quad y = \frac{\text{det}(A_y)}{\text{det}(A)}, \quad z = \frac{\text{det}(A_z)}{\text{det}(A)} $$
where $A_x$, $A_y$, and $A_z$ are matrices formed by replacing the respective column of matrix $A$ with vector $B$.

In [32]:
import copy
def solve_cramer(matrix: list[list[float]], vector: list[float]) -> list[float]:
    n = len(matrix)
    detA = determinant(matrix)
    if detA == 0:
        raise ValueError("No unique solution")
    
    result = []
    for i in range(n):
        Ai = copy.deepcopy(A)
        for row in range(n):
            Ai[row][i] = vector[row]
        result.append(determinant(Ai) / detA)
    return result

print(f"{solve_cramer(A, B)=}")

solve_cramer(A, B)=[1.4285714285714286, -0.42857142857142855, 0.6428571428571429]


### **4. Solving using Inversion (3 points)**

Finally, solve the system by computing the inverse of matrix $A$ and multiplying it by vector $B$.
$$ A \cdot X = B \rightarrow X = A^{-1} \cdot B $$
**Adjugate Method for Matrix Inversion:**

To find the inverse of matrix $ A $, you can use the adjugate method:
$$ A^{-1} = \frac{1}{\text{det}(A)} \times \text{adj}(A) $$
where $\text{adj}(A)$ is the adjugate (or adjoint) matrix, which is the transpose of the cofactor matrix of $ A $.

**Cofactor Matrix:**

The cofactor matrix is a matrix where each element is replaced by its cofactor. The cofactor of an element $a_{ij}$ is given by:
$$ (-1)^{i+j} \times \text{det}(M_{ij}) $$
where $M_{ij}$ is the minor of element $a_{ij}$, which is the matrix obtained by removing the $i$-th row and $j$-th column from matrix $A$.

In [46]:
def minor(matrix: list[list[float]], i: int, j: int) -> list[list[float]]:
    minor_matrix = []
    line = []
    n = len(matrix)
    for i1 in range(n):
        if i1 == i:
            continue
        for j1 in range(n):
            if j1 == j:
                continue
            line.append(matrix[i1][j1])
        minor_matrix.append(line)
        line = []
    return minor_matrix

def cofactor(matrix: list[list[float]]) -> list[list[float]]:
    n = len(matrix)
    cofactor_matrix = [[0.0 for _ in range(n)] for _ in range(n)]
    for i in range(n):
        for j in range(n):
            m = minor(matrix, i, j)
            cofactor_matrix[i][j]=((-1)**(i+j))*determinant(m)
    return cofactor_matrix

def adjoint(matrix: list[list[float]]) -> list[list[float]]:
    cof = cofactor(matrix)
    return transpose(cof)

def solve(matrix: list[list[float]], vector: list[float]) -> list[float]:
    n = len(matrix)
    inverse_matrix = adjoint(matrix)
    for i in range(n):
        for j in range(n):
            inverse_matrix[i][j] *= 1/determinant(matrix)

    return multiply(inverse_matrix, vector)

print(f"{solve(A, B)=}")
print(minor(A, 1, 1))
print(cofactor(A))

solve(A, B)=[0.35714285714285765, 2.071428571428571, 1.9285714285714293]
[[2.0, -1], [3.0, 2.0]]
[[-6.0, 10.0, 4.0], [-7.0, 7.0, 7.0], [11.0, -9.0, -5.0]]
